<a href="https://colab.research.google.com/github/oliviadellaglio/ds2002-fa26/blob/main/03-pandas-cleaning/2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df["revenue"] = df["qty"] * df["price"] #creates a new column 'revenue', calculated from the qty * price for each row
total_revenue = sum(df["revenue"]) #sums up column
print('Total revenue: $', total_revenue) #prints total revenue
print('Number of units: ', sum(df["qty"])) #prints number of total units

Total revenue: $ 8520.0
Number of units:  783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
rev_summary = (df.groupby("category").agg(revenue = ('revenue', 'sum')))
#groups rows by category and adds up revenue for each category

rev_summary['share_of_revenue'] = rev_summary["revenue"] / rev_summary["revenue"].sum() * 100
#divides each category revenue by total revenue and multiplies by 100 to make percentage

rev_summary = (rev_summary.round(2).sort_values('revenue', ascending = False))
#rounds to 2 decimal places and sorts from largest to smallest

rev_summary
#prints table



,revenue,share_of_revenue
category,,
Food,4293.0,50.39
Merch,1771.5,20.79
Drink,1554.0,18.24
RainGear,901.5,10.58


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
vend_summary = (df.groupby("vendor_id").agg(orders = ('vendor_id', 'count'), average_revenue = ('revenue', 'mean')))
#groups by vendor_id, sums up forders for each vendor, calculates the average revue for each vendor

vend_summary = (vend_summary.round(2).sort_values('average_revenue', ascending = False))
#rounds values to 2 decimal places and sort from highest to lowest by average revnue

vend_summary
#print table


,orders,average_revenue
vendor_id,,
V-01,94,22.60
V-18,108,21.75
V-05,93,20.58
V-10,105,20.31


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_share = rev_summary.loc["Merch", "share_of_revenue"].round(1)
#pulls percentage of revnue from merch from rev_summary table and rounds to 1 decimal place

print(merch_share, "% of revenue comes from Merch.")
#prints revenue percentage

20.8 % of revenue comes from Merch.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on="vendor_id", how = 'left', validate = 'many_to_one', indicator = True)
#join vendor_names onto df on vendor_id using left join

print('rows before:', len(df), '| rows after: ', len(joined))
print()
print(joined['_merge'].value_counts())
joined[['vendor_id', 'vendor_name', 'qty', 'price', '_merge']]
#check that it worked--rows before = rows after

missing_vendor = joined.loc[joined["_merge"] == "left_only", ["vendor_id", "vendor_name"]]
missing_vendor_id = missing_vendor.iloc[0]["vendor_id"]
#find vendor missing vendor id and store the name as a variable

print(missing_vendor_id, " is does not have a vendor name.")
#print missing vendor id

joined.loc[joined["_merge"] == "left_only", "vendor_name"] = "Unknown"
#set vendor name for vendor id with missing vendor name as "Unknown"

joined
#print table


rows before: 400 | rows after:  400

_merge
both          292
left_only     108
right_only      0
Name: count, dtype: int64
V-18  is does not have a vendor name.


,vendor_id,category,qty,price,revenue,vendor_name,_merge
0,V-10,Drink,2,24.0,48.0,Cav Merch North,both
1,V-18,RainGear,1,12.0,12.0,Unknown,left_only
2,V-18,Drink,3,4.5,13.5,Unknown,left_only
3,V-10,Food,2,12.0,24.0,Cav Merch North,both
4,V-18,Drink,3,7.5,22.5,Unknown,left_only
...,...,...,...,...,...,...,...
395,V-18,Merch,1,12.0,12.0,Unknown,left_only
396,V-01,Merch,2,24.0,48.0,Hoos Burgers,both
397,V-10,Food,3,7.5,22.5,Cav Merch North,both
398,V-18,Merch,2,24.0,48.0,Unknown,left_only


**The unmatched vendor, and what I did about it:** _..._

*   The unmatched vendor is V-18
*   I set vendor name as "Unknown"



### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot = pd.pivot_table(joined, index="vendor_name", columns="category", values = "revenue", aggfunc="sum", margins = True, margins_name = "Total")
#create pivot table from joined data frame with vendor names down the side, columns across the top, and add up revenues for each cell

pivot
#print pivot table

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(rev_summary['revenue'].sum() - df['revenue'].sum()) < 0.01
#changed by_category to rev_summary (the variable I used in Q2)
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.


**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) Next game, vendors should focus on the areas where the revenue was highest. For example, the category with the highest reneue was Food, which accounted for 50.39% of total revenue. Hoos Burgers was the vendor with the highest food revenue, bringing in $1338. Hoos Burgers should prioritize stocking things like burger patties, buns, condiments, etc. Rain gear was the worst perfroming category, only accounting for 10.58% of the revenue. Next game, vendors should decrease the stock of the rain gear. Potentially, they could decrease the number of stands selling rain gear in order to decrease the amount of employees being paid to sell rain gear. This could potentially be changed when there is rain forecasted, because weather will likely determine how much revenue is generated from rain gear.

b) My answer for Q3 determining highest average revnue is least trustworthy. This is because there are only about 100 orders per vendor to base the average on. The small sample size makes it difficult to determine wether or not the top vendor has consistantly higher value orders.
